# Embedding overlap analysis

In [1]:
import pandas as pd
import numpy as np
import random as rd
from sklearn.neighbors import NearestNeighbors
from typing import List
from math import log2

## 1. Read embedding files

In [ ]:
title_df = pd.read_csv('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.title', sep='\t')
title_df.head(5)

In [3]:
image_df = pd.read_csv('../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.image', sep='\t')
image_df.head(5)

,img_id:token,img_emb:float_seq
0,884509,2.116241893624543 -3.0418755567230007 -5.46931...
1,561856,1.649818936829127 3.1124404940420742 2.8051456...
2,239749,-1.578862879367632 -4.014066168838383 0.149213...
3,55030,3.093842411693621 -1.7237219859956285 -0.78452...
4,1277121,-1.2830992703265296 5.067334132483662 1.839873...


## 2. Preprocess embeddings

In [4]:
title_df.rename(columns={"ent_id:token": "id", "ent_emb:float_seq": "emb"}, inplace=True)
title_df["emb"] = title_df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
title_df.head(5)

,id,emb
0,0000032069,"[-0.12467234, 0.14852484, -0.18677841, 0.24061..."
1,0000031909,"[-0.07143591, 0.19644818, -0.18831302, 0.26631..."
2,0000032034,"[-0.05860839, 0.19564873, -0.1866356, 0.225499..."
3,0000031852,"[-0.12591696, 0.16939871, -0.16362002, 0.26739..."
4,0000032050,"[-0.07276042, 0.15790446, -0.14320622, 0.22367..."


In [5]:
image_df.rename(columns={"img_id:token": "id", "img_emb:float_seq": "emb"}, inplace=True)
image_df["emb"] = image_df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))
image_df.head(5)

,id,emb
0,884509,"[2.116242, -3.0418756, -5.469318, 0.9049815, -..."
1,561856,"[1.6498189, 3.1124406, 2.8051457, -1.02009, -1..."
2,239749,"[-1.5788629, -4.014066, 0.14921321, 1.5560805,..."
3,55030,"[3.0938425, -1.723722, -0.78452605, -1.4718788..."
4,1277121,"[-1.2830993, 5.067334, 1.8398732, 0.92253864, ..."


## 3. Fit KNN with static embeddings

In [6]:
X = np.vstack(title_df["emb"].values)
title_knn = NearestNeighbors(n_neighbors=100, metric='cosine')
title_knn.fit(X)

NearestNeighbors(metric='cosine', n_neighbors=100)

In [7]:
Y = np.vstack(image_df["emb"].values)
image_knn = NearestNeighbors(n_neighbors=100, metric='cosine')
image_knn.fit(Y)

NearestNeighbors(metric='cosine', n_neighbors=100)

## 4. Take random item and find its neighbours

In [62]:
assert title_df.shape[0] == image_df.shape[0]

random_ids = []
while True:
    random_ids = rd.sample(range(title_df.shape[0]), 10)
    if all(sum(X[random_id]) != 0 and sum(Y[random_id]) != 0 for random_id in random_ids):
        break

for random_id in random_ids:
    assert title_df["id"].iloc[random_id] == image_df["id"].iloc[random_id]

original_ids = [title_df["id"].iloc[random_id] for random_id in random_ids]
title_distance, title_nearest_ids = title_knn.kneighbors([X[random_id] for random_id in random_ids])
image_distance, image_nearest_ids = image_knn.kneighbors([Y[random_id] for random_id in random_ids])

## 5. Metrics

In [11]:
def iou_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 0.0

def recall_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a) if set_a else 0.0

def dcg_at_k(relevance: List[int], k: int) -> float:
    return sum(rel / log2(idx + 2) for idx, rel in enumerate(relevance[:k]))

def ndcg_between(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    ideal_relevance = [k - i for i in range(min(k, len(neighbors_a)))]
    position_map = {cid: i for i, cid in enumerate(neighbors_a[:k])}
    relevance_b = []
    for cid in neighbors_b[:k]:
        if cid in position_map:
            relevance_b.append(k - position_map[cid])
        else:
            relevance_b.append(0)
    dcg = dcg_at_k(relevance_b, k)
    ideal_dcg = dcg_at_k(ideal_relevance, k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

In [64]:
metric_df = pd.DataFrame(columns=["id", "iou@100", "recall@100", "ndcg@100"])
for i in range(len(title_nearest_ids)):
    iou = f"{iou_at_k(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
    recall = f"{recall_at_k(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
    ndcg = f"{ndcg_between(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
    metric_df.loc[len(metric_df)] = [original_ids[i], iou, recall, ndcg]
metric_df["id"] = metric_df["id"].astype(int)
metric_df.head(10)

,id,iou@100,recall@100,ndcg@100
0,866164,2.04%,4.00%,15.41%
1,1069621,14.94%,26.00%,33.50%
2,940836,57.48%,73.00%,83.34%
3,306294,1.01%,2.00%,12.94%
4,1211511,0.50%,1.00%,8.00%
5,1479898,7.53%,14.00%,40.26%
6,85853,6.38%,12.00%,31.02%
7,1289807,3.09%,6.00%,12.14%
8,777085,1.01%,2.00%,9.33%
9,254613,5.26%,10.00%,25.67%


## 6. Analysis for whole dataset (16h is too much)

In [13]:
from tqdm import tqdm
original_ids = list(image_df['id'])
ids = range(len(original_ids))
BATCH_SIZE = 100
L = 0
R = BATCH_SIZE

metric_df = pd.DataFrame(columns=["id", "iou@100", "recall@100", "ndcg@100"])

for _ in tqdm(range(len(ids) // BATCH_SIZE)):
    ids_slice = ids[L:R]

    title_distance, title_nearest_ids = title_knn.kneighbors([X[idx] for idx in ids_slice])
    image_distance, image_nearest_ids = image_knn.kneighbors([Y[idx] for idx in ids_slice])

    

    for i in range(len(title_nearest_ids)):
        iou = f"{iou_at_k(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
        recall = f"{recall_at_k(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
        ndcg = f"{ndcg_between(title_nearest_ids[i], image_nearest_ids[i], 100) * 100:.2f}%"
        metric_df.loc[len(metric_df)] = [original_ids[i], iou, recall, ndcg]

    L = R
    R += BATCH_SIZE

    if R == len(original_ids):
        break

    if R > len(original_ids):
        R = len(original_ids)

metric_df["id"] = metric_df["id"].astype(int)
metric_df.head(10)

  0%|          | 13/15872 [00:49<16:38:31,  3.78s/it]


KeyboardInterrupt: 